# 04 — Effects and the ATT

**Goal.** Estimate the treatment effect several ways, for both pools, and score each one
against the experimental benchmark.

| | |
|---|---|
| **Reads** | `outputs/data/02_scored_*.csv`, `03_pairs_*.csv`, `01_benchmark.json` |
| **Writes** | `outputs/tables/results.csv`, plus the cover figure |

**Questions this notebook answers**

1. What is the matched ATT for each pool, with a confidence interval?
2. Do IPW (with ATT weights) and the doubly robust estimator agree with it?
3. Does each confidence interval cover the benchmark?
4. How wide are the intervals, and what does that rule in or out?

Report the point estimate, the interval, and the conclusion together. The benchmark's own
interval spans about $2,600, so with 185 treated units no method here can be more precise than
that. A point estimate landing within a few dollars of the truth is luck, not evidence.

Estimand discipline matters in this notebook. Matching targets the ATT. IPW targets whatever
its weights say: use `estimand="ATT"` so the two are comparable, and treat the ATE as a
separate question rather than a robustness check.

The cover figure is `figures.forest_plot`, left unimplemented on purpose: it is the one plot
that has to communicate the whole project to someone with no statistics background.

In [ ]:
import sys
sys.path.insert(0, "..")

# Reload src/ modules automatically whenever they change on disk. Without this,
# Python caches the module on first import and later edits to src/*.py are
# invisible until you restart the kernel.
%load_ext autoreload
%autoreload 2

import json
import warnings

warnings.filterwarnings("ignore", message=".*numexpr.*")
warnings.filterwarnings("ignore", message=".*bottleneck.*")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import psm, data, figures

pd.set_option("display.width", 200)
pd.set_option("display.float_format", "{:,.2f}".format)

POOLS = ["cps", "psid"]

%matplotlib inline

In [ ]:
benchmark = json.loads((data.OUT_DIR / "01_benchmark.json").read_text())
scored = {pool: data.load_stage(f"02_scored_{pool}") for pool in POOLS}
pairs  = {pool: data.load_stage(f"03_pairs_{pool}") for pool in POOLS}

print(f"benchmark ATT ${benchmark['estimate']:,.0f} "
      f"[{benchmark['ci_lo']:,.0f}, {benchmark['ci_hi']:,.0f}]\n")
for pool in POOLS:
    print(f"{pool:5s} scored {len(scored[pool]):>6,} rows   matched pairs {len(pairs[pool]):>4,}"
          f"   of {int(scored[pool].treat.sum())} treated")